## CS310 Natural Language Processing
## Assignment 1. Neural Network-based Text Classification

**Total points**: 30

You should roughtly follow the structure of the notebook. Add additional cells if you feel needed. 

You can (and you should) re-use the code from Lab 2. 

Make sure your code is readable and well-structured.

### 0. Import Necessary Libraries

In [2]:
import json
import re
import jieba
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

#检测GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


/data/student/.conda/envs/fjm_MLP/lib/python3.12/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Using device: cuda


### 1. Data Processing

In [3]:
def basic_tokenizer(text):
    #使用正则表达式匹配所有中文字符，单字切分并丢弃其他所有符号
    #\u4e00-\u9fa5是汉字Unicode编码范围
    chinese_chars = re.findall(r'[\u4e00-\u9fa5]', text)
    return chinese_chars
def advanced_tokenizer(text):
    #使用jieba按词进行切分，保留数字标点等
    return jieba.lcut(text)

# 读取语料文件
def load_data(filepath):
    texts, labels = [], []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            texts.append(data['sentence'])
            labels.append(data['label'][0]) #保证label是0/1
    return texts, labels

train_texts, train_labels = load_data('train.jsonl')
test_texts, test_labels = load_data('test.jsonl')
print(f"载入训练数据 {len(train_texts)} 条，测试数据 {len(test_texts)} 条")
# 切换分词器,False为基本单字分词，True为jieba分词
USE_ADVANCED_TOKENIZER = False
tokenizer = advanced_tokenizer if USE_ADVANCED_TOKENIZER else basic_tokenizer
# 根据训练集构建词表
word_counts = Counter()
for text in train_texts:
    word_counts.update(tokenizer(text))

# 建立词表映射（word -> index）
# <unk> 用于映射在测试集中出现、但在训练集中从未见过的生词
vocab = {'<pad>': 0, '<unk>': 1}
for word, _ in word_counts.items():
    vocab[word] = len(vocab)
print(f"目前使用的分词器: {'高级分词(jieba)' if USE_ADVANCED_TOKENIZER else '基础分字(正则)'}")
print(f"产生的Vocabulary Size: {len(vocab)}")
# 将纯文本句子转化成整数索引列表
def text_to_indices(text):
    tokens = tokenizer(text)
    return [vocab.get(token, vocab['<unk>']) for token in tokens]

#构建Dataloader
class HumorDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        indices = text_to_indices(self.texts[idx])
        # 文本为 int 类型序列，标签为 float(二分类需要)
        return torch.tensor(indices, dtype=torch.int64), torch.tensor(self.labels[idx], dtype=torch.float32)
train_dataset = HumorDataset(train_texts, train_labels)
test_dataset = HumorDataset(test_texts, test_labels)
# 对于nn.EmbeddingBag需实现自定义collate_fn。
# 因为每个句子长度不同，我们需要将一个 batch 里的所有词典按行拼接，并记录各句子的起始点 offset
def collate_fn(batch):
    labels, text_list, offsets = [], [], [0]
    
    for text_indices, label in batch:
        labels.append(label)
        text_list.append(text_indices)
        offsets.append(text_indices.size(0))
        
    labels = torch.tensor(labels, dtype=torch.float32)
    offsets = torch.tensor(offsets[:-1]).cumsum(dim=0) # 计算前缀和作为序列边界
    text_tensor = torch.cat(text_list) # 将所有短句拼接为一条1维张量
    
    return text_tensor, labels, offsets
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

载入训练数据 12677 条，测试数据 651 条
目前使用的分词器: 基础分字(正则)
产生的Vocabulary Size: 2688


### 2. Build the Model

In [4]:
class HumorClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim1, hidden_dim2, num_class):
        super(HumorClassifier, self).__init__()
        
        #EmbeddingBag (使用平均模式将句子变长词向量平摊压缩成定长)
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, sparse=False, mode='mean')
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim1),
            nn.ReLU(),
            nn.Dropout(0.3),  #Dropout避免过拟合
            
            #第一层 Hidden layer
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            #第二层 Hidden layer
            #由于当前是二分类，输出1维并通过一个 Sigmoid 操作限定为概率 0~1 的值
            nn.Linear(hidden_dim2, num_class),
            nn.Sigmoid()
        )
    def forward(self, text, offsets):
        embedded = self.embedding(text, offsets)
        output = self.fc(embedded)
        #[batch_size, 1]->[batch_size]
        return output.squeeze()


#设置超参数+模型实例化
VOCAB_SIZE = len(vocab)
EMBED_DIM = 64
HIDDEN_DIM1 = 32
HIDDEN_DIM2 = 16
NUM_CLASS = 1 # 二分类输出的通道是1维

model = HumorClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM1, HIDDEN_DIM2, NUM_CLASS).to(device)
print(model)

HumorClassifier(
  (embedding): EmbeddingBag(2688, 64, mode='mean')
  (fc): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=16, out_features=1, bias=True)
    (7): Sigmoid()
  )
)


### 3. Train and Evaluate

In [5]:
criterion = nn.BCELoss() #二元交叉熵损失
optimizer = optim.Adam(model.parameters(), lr=0.001)
EPOCHS = 10
print("==== Starts Training ====")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    
    for texts, labels, offsets in train_loader:
        texts, labels, offsets = texts.to(device), labels.to(device), offsets.to(device)
        
        optimizer.zero_grad() # 清空梯度
        predicted = model(texts, offsets) # 正向传播
        loss = criterion(predicted, labels) # 测定损失
        loss.backward() # 反向传播
        optimizer.step() # 参数更新
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch: [{epoch}/{EPOCHS}], Average Loss: {avg_loss:.4f}")
print("\n==== Starts Evaluation ====")
model.eval()
#保存真实标签和模型的预测标签（由概率转换成了类别 0/1）
all_preds = []
all_labels = []
with torch.no_grad():
    for texts, labels, offsets in test_loader:
        texts, labels, offsets = texts.to(device), labels.to(device), offsets.to(device)
        predicted_probs = model(texts, offsets)
        
        #大于 0.5 视为分类为 1(幽默)，反之为 0
        preds = (predicted_probs >= 0.5).int().cpu().numpy()
        
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
# 利用 sklearn 计算评测指标
acc = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, zero_division=0)
recall = recall_score(all_labels, all_preds, zero_division=0)
f1 = f1_score(all_labels, all_preds, zero_division=0)
print(f"Test Accuracy:  {acc:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall:    {recall:.4f}")
print(f"Test F1 Score:  {f1:.4f}")

==== Starts Training ====
Epoch: [1/10], Average Loss: 0.5938
Epoch: [2/10], Average Loss: 0.5788
Epoch: [3/10], Average Loss: 0.5699
Epoch: [4/10], Average Loss: 0.5611
Epoch: [5/10], Average Loss: 0.5502
Epoch: [6/10], Average Loss: 0.5402
Epoch: [7/10], Average Loss: 0.5277
Epoch: [8/10], Average Loss: 0.5197
Epoch: [9/10], Average Loss: 0.5113
Epoch: [10/10], Average Loss: 0.5003

==== Starts Evaluation ====
Test Accuracy:  0.7066
Test Precision: 0.4234
Test Recall:    0.3412
Test F1 Score:  0.3779
